# 02.01 · What is ML? Supervised, unsupervised, splits

**Chapter:** Machine Learning Fundamentals · **Milestone:** M1

## Goals
- [ ] Distinguish supervised, unsupervised and self-supervised learning (how LLMs learn)
- [ ] Always start from a baseline
- [ ] Split train / validation / test correctly and recognise data leakage

## Resources
- [Google ML Crash Course](https://developers.google.com/machine-learning/crash-course) — free
- [scikit-learn – Common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html) — free

**Machine learning** = instead of writing rules, show examples and let an algorithm fit a function $f(x) \approx y$.

| Type | Data | Example | In AI engineering |
|---|---|---|---|
| Supervised | inputs + labels | spam / not spam | classifiers, rerankers, LLM judges' calibration |
| Unsupervised | inputs only | clustering, PCA | grouping user queries, visualising embeddings |
| Self-supervised | raw data; labels come *from the data itself* | predict the next word | **how every LLM is pretrained** |
| Reinforcement | actions + rewards | games, RLHF | post-training chat & reasoning models |

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Self-supervised: labels for free
Any text becomes a supervised dataset: input = tokens so far, label = next token. No human labelling.
This is why LLMs could be trained on trillions of tokens.

In [ ]:
text = "the cat sat on the mat".split()
pairs = [(text[:i], text[i]) for i in range(1, len(text))]
for context, target in pairs:
    print(f"{' '.join(context):>22} -> {target}")

## 2. A supervised task, with a baseline first
Breast-cancer dataset: 30 numeric features per tumour, label malignant/benign.
**Always** compute a dumb baseline (predict the majority class). A model that doesn't beat it is useless — common with LLM
classifiers on imbalanced data, too.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
print(X.shape, "| class balance:", np.bincount(y) / len(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print(f"baseline accuracy: {baseline.score(X_test, y_test):.3f} | logistic regression: {model.score(X_test, y_test):.3f}")

## 3. Train / validation / test
- **Train**: fit parameters.
- **Validation**: choose hyperparameters, prompts, models. You look at it often.
- **Test**: touch **once** at the end for an honest estimate. If you tune on it, it's no longer a test set.

Same rule later for LLM apps: your eval set is your validation set; keep a held-out set to avoid overfitting your prompts.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=0, stratify=y_tmp)
print(len(X_train), len(X_val), len(X_test))

best = max(
    (make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000)).fit(X_train, y_train) for C in [0.01, 0.1, 1, 10]),
    key=lambda m: m.score(X_val, y_val),
)
print("chosen C:", best[-1].C, "| val:", round(best.score(X_val, y_val), 3), "| test (once!):", round(best.score(X_test, y_test), 3))

## 4. Data leakage: the silent killer
Leakage = information from the test set sneaks into training. Results look great, production fails.
Demo: **pure random noise** features and random labels. Honest accuracy must be ≈ 50%.
But if we select the "most predictive" features using *all* the data before splitting…

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 10_000))
y_noise = rng.integers(0, 2, 200)

# WRONG: feature selection sees all labels, then we cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise, cv=5).mean()

# RIGHT: selection happens inside each training fold via a pipeline
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), X_noise, y_noise, cv=5).mean()
print(f"leaky CV accuracy on pure noise: {leaky:.2f}  | honest: {honest:.2f}")

Other leakage you'll meet: duplicates across train/test, future data in time series, and — for LLMs — **benchmark
contamination** (test questions in the pretraining data) and eval examples copied into few-shot prompts.

## Exercises
1. Reproduce leakage another way: fit `StandardScaler` on the full dataset before splitting. Does it matter much here? Why is it still wrong?
2. Make the breast-cancer data imbalanced (keep only 20 malignant cases). What happens to the baseline's accuracy? Is accuracy still a good metric? (Preview of 02.03.)
3. Write 3 examples of self-supervised objectives besides next-token prediction (hint: masked words, image patches, contrastive pairs).

In [ ]:
# your code here